In [1]:
# IMPORT LIBRARY
import pandas as pd
import numpy as np
from scipy.stats import norm
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
# MENDEFINISIKAN FUNGSI MATEMATIS BLACK-SCHOLES
def black_scholes_put(S, K, T, r, sigma):
    intrinsic = np.maximum(K - S, 0)

    sqrtT = np.sqrt(T)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    price = np.where(T == 0, intrinsic, price)

    return price

In [3]:
# PENILAIAN HARGA TEORITIS OPSI PUT
df = pd.read_csv("Data_Final_Model_Black-Scholes.csv")

df["DTE"] = df["DTE"] / 365
df["r_KONSTAN"] = df["r_KONSTAN"] / 100

df["BS_PRICE"] = black_scholes_put(
    df["UNDERLYING_LAST"],  
    df["STRIKE"],           
    df["DTE"],              
    df["r_KONSTAN"],             
    df["VOLATILITAS_KONSTAN"]    
)
    
rmse = np.sqrt(mean_squared_error(df["MID_PRICE"], df["BS_PRICE"]))
mae  = mean_absolute_error(df["MID_PRICE"], df["BS_PRICE"])

result = df[[
    "ID_OPTION",
    "QUOTE_DATE",
    "MID_PRICE",
    "BS_PRICE"
]].copy()

result.to_csv("Penilaian_Model_Black-Scholes.csv",index=False)